# 🤖 PCB RAG Testing Notebook

This notebook demonstrates:
- Converting PCB designs → documents
- Building a vector store
- Querying designs
- Generating answers using LLM (Groq optional)

---
Future: integrate LangChain / LangGraph

In [ ]:
# Setup
from scripts.ingest_rag import design_to_docs, SimpleVectorStore

import json
import pprint

# Optional LLM (Groq)
USE_LLM = False

if USE_LLM:
    from groq import Groq
    import os
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## 🧩 Sample PCB Designs

In [ ]:
designs = [
    {
        "components": [
            {"ref": "R1", "value": "10k"},
            {"ref": "C1", "value": "100nF"}
        ],
        "nets": [
            {"name": "VCC", "connections": ["R1:1", "C1:1"]}
        ]
    },
    {
        "components": [
            {"ref": "U1", "value": "ATmega328"}
        ],
        "nets": [
            {"name": "GND", "connections": ["U1:GND"]}
        ]
    }
]

## 📄 Convert Designs → Documents

In [ ]:
docs = []

for d in designs:
    docs.extend(design_to_docs(d))

pprint.pprint(docs)

## 🗄 Build Vector Store

In [ ]:
store = SimpleVectorStore()

for doc in docs:
    store.add(doc["text"], doc["metadata"])

store.data

## 🔍 Simple Retrieval

In [ ]:
def retrieve(query, store):
    return [d for d in store.data if query.lower() in d["text"].lower()]

results = retrieve("resistor", store)

results

## 📊 Ranking Results

In [ ]:
def rank_results(query, results):
    return sorted(results, key=lambda x: query.lower() in x["text"].lower(), reverse=True)

ranked = rank_results("capacitor", store.data)
ranked

## 🧠 Prompt Construction

In [ ]:
def build_prompt(query, docs):
    context = "\n".join([d["text"] for d in docs])

    return f"""
You are a PCB design assistant.

Context:
{context}

Question:
{query}

Answer clearly:
"""

prompt = build_prompt("What components are present?", docs)
print(prompt)

## 🤖 LLM Query (Groq - Optional)

In [ ]:
if USE_LLM:
    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    print(response.choices[0].message.content)
else:
    print("LLM disabled. Set USE_LLM=True to enable.")

## 💬 Interactive Query

In [ ]:
query = input("Ask about PCB design: ")

results = retrieve(query, store)
prompt = build_prompt(query, results)

print("\n--- Retrieved Context ---")
pprint.pprint(results)

if USE_LLM:
    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    print("\n--- LLM Answer ---")
    print(response.choices[0].message.content)
else:
    print("\n--- Prompt ---")
    print(prompt)

## 🚀 Future Improvements

- Replace SimpleVectorStore with FAISS / Chroma
- Add embeddings (OpenAI / HuggingFace)
- Use LangChain RetrievalQA
- Add memory (chat history)
- Integrate with LangGraph agents
- Add PCB-specific reasoning prompts